## 1.

Datasets:

    SPRINGLab/IndicVoices-R_Tamil
    SPRINGLab/IndicVoices-R_Hindi
    SPRINGLab/IndicVoices-R_Bengali

Data Preparation Steps:

    Load Datasets: For each of the three datasets (Tamil, Hindi, Bengali), load the train split.
    Create Subsets: Take the first 500 examples from the train split of each dataset. This will give you three subsets, one for each language.
    Add the lang column to each subsets and assign Language Labels:
        Label all Tamil examples with 0.
        Label all Hindi examples with 1.
        Label all Bengali examples with 2.
    Combine Subsets: Combine the three language subsets into a single list.
    Shuffle the Combined Data: Shuffle the combined list of examples randomly.

Reproducibility:

    Set Random Seed: To ensure consistent results when shuffling the combined data, use a random seed of 42.

What is the sampling rate (in Hz) of the audio data?

In [ ]:
import pandas as pd

In [ ]:
from datasets import load_dataset

# Load Datasets
tamil_ds = load_dataset("SPRINGLab/IndicVoices-R_Tamil", split="train",streaming=True)
hindi_ds = load_dataset("SPRINGLab/IndicVoices-R_Hindi", split="train",streaming=True)
bengali_ds = load_dataset("SPRINGLab/IndicVoices-R_Bengali", split="train",streaming=True)

In [ ]:
tamil_subset=[]
hindi_subset=[]
bengali_subset=[]

In [ ]:
for i, example in enumerate(tamil_ds):
  if i<500:
    tamil_subset.append(example)
  else:
    break

for i,example in enumerate(hindi_ds):
  if i<500:
    hindi_subset.append(example)
  else:
    break

for i,example in enumerate(bengali_ds):
  if i<500:
    bengali_subset.append(example)
  else:
    break

In [ ]:
tamil_subset[0]

In [ ]:
import matplotlib.pyplot as plt

# Suppose your dataset entry is stored in variable `example`
audio = tamil_subset[0]["audio"]   # This is the AudioDecoder object
waveform = audio["array"]
sample_rate = audio["sampling_rate"]

# Play audio (in Jupyter/Colab you can use IPython.display)
from IPython.display import Audio
Audio(waveform, rate=sample_rate)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(waveform)
plt.title("Audio Waveform")
plt.xlabel("Samples")
plt.ylabel("Amplitude")
plt.show()

In [ ]:
audio["array"]

In [ ]:
for i, example in enumerate(tamil_subset):
  example["lang"] = 0

for i, example in enumerate(hindi_subset):
  example["lang"] = 1

for i, example in enumerate(bengali_subset):
  example["lang"] = 2

In [ ]:
data = tamil_subset + hindi_subset + bengali_subset

In [ ]:
import random
random.seed(42)
random.shuffle(data)

In [ ]:
data[400]["lang"]

In [ ]:
print(f"The sampling rate of the dataset is {data[0]['audio']['sampling_rate']}")

## Q2. What is the speaker_id of the 15th data point in the Hindi subset before shuffling?

In [ ]:
hindi_subset[15]["speaker_id"]

## Common Information for Questions 3 to 8

    Model and Processor:
        Load the Wav2Vec 2.0 model (facebook/wav2vec2-base-960h) and its processor.
    Audio Processing:
        Use the processor to prepare the audio data with a sampling rate of 16,000 Hz.
        Use the options: return_tensors="pt" to get PyTorch tensors and padding=True to pad the audio if necessary.
    Feature Extraction:
        Pass the processed audio to the Wav2Vec 2.0 model ensuring it outputs hidden states.
        Extract features from the 9th and 11th layers of the Wav2Vec2 model.
        Average these two layers to obtain the final feature representation.
    Batch Size:
        Use a batch_size of 4 during feature extraction.

What is the shape of the feature vector extracted from a single audio sample after processing with the Wav2Vec2.0 model and averaging the specified layers?

In [ ]:
import torch
from transformers import Wav2Vec2Model, Wav2Vec2Processor

# Load the Wav2Vec 2.0 model and its processor
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h")

In [ ]:
call_count = 0

def extract_word2vec_features(batch):
  global call_count
  call_count += 1
  print(f"Call count: {call_count}")

  # Extract the audio arrays from the AudioDecoder objects
  audio_arrays = [item['audio']['array'] for item in batch]

  inputs = processor(audio_arrays, sampling_rate=16000, return_tensors="pt", padding=True)
  with torch.no_grad():
    model_outputs = model(**inputs, output_hidden_states=True)
    layer9 = model_outputs.hidden_states[8]
    layer11 = model_outputs.hidden_states[10]
    outputs = (layer9 + layer11) / 2
    if call_count == 1:
      layer7 = model_outputs.hidden_states[6]
      layer12 = model_outputs.hidden_states[11]
      print(((layer7+layer12)/2).mean(dim=1).numpy())
  label = [item["lang"] for item in batch]
  return {"features": outputs.mean(dim=1).numpy(), "labels": label}

In [ ]:
from tqdm.auto import tqdm

subset_with_features = []
batch_size = 4
for i in tqdm(range(0, len(data), batch_size)):
    batch = data[i:i+batch_size]
    processed_batch = [ {'audio': item['audio'], 'lang': item['lang']} for item in batch] # Reformat batch for feature extraction function
    features_output = extract_word2vec_features(processed_batch)

    # Iterate through the extracted features and labels and append them individually
    extracted_features = features_output['features']
    extracted_labels = features_output['labels']
    for j in range(len(extracted_features)): # Iterate through the features (which is the same length as labels)
        subset_with_features.append({'features': extracted_features[j], 'label': extracted_labels[j]})

## Q.No - 4

If you extract features for all 1500 audio samples in the subset_with_features list, what would be the shape of the resulting feature matrix?

In [ ]:
subset_with_features[0]['features'].shape

## Q.No - 5
How many parameters (in millions) does the Wav2Vec2.0 base model (facebook/wav2vec2-base-960h) have in total? Report your answer rounded to two decimal places.

In [ ]:
model.num_parameters() / 1e6

## Q.No - 6

What type of layers are used in the encoder part of the Wav2Vec2.0 base model architecture?

Ans : Convolution Layer

## Q.No - 7
Modify the extract_wav2vec_features function to average the hidden states from the 7th and 12th layers instead of the 9th and 11th. After running the modified code, what is the value of the first element of the feature vector extracted for the first audio sample in your subset? Report the answer rounded to four decimal places.

Ans : 0.025

## Q.No - 8
What is the gender of the 25th data point in the Tamil streaming dataset (before shuffling)?

In [ ]:
tamil_ds[25]["gender"]

## Q.No - 9 & 10

    Data Splitting:
        Use train_test_split (with random_state=42) from sklearn to split the subset_with_features dataset into training and testing sets.
        Use a test size of 20%.
        Stratify by the 'label' to ensure language class balance in both sets.

    Model Training:
        Train a Logistic Regression model using the extracted features to classify the language (Tamil, Hindi, Bengali).
        Use LogisticRegression(random_state=42, max_iter=1000).

What is the validation accuracy (on a 20% validation split from the training data, with random_state=42 and stratified by label) of the Logistic Regression model after training on the extracted features? Enter your answer in percentage.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import numpy as np


features = [item['features'] for item in subset_with_features]
labels = [item['label'] for item in subset_with_features]

X = np.array(features)
y = np.array(labels)

# Train-Test Split (Stratified)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Train-Validation Split (from training set, Stratified)
X_train_new, X_val, y_train_new, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train) # Validation 20% of training

# Train Logistic Regression
logistic_model = LogisticRegression(random_state=42, max_iter=1000) # Increased max_iter for convergence
logistic_model.fit(X_train_new, y_train_new)

# Validation Accuracy
y_val_pred = logistic_model.predict(X_val)
val_accuracy = accuracy_score(y_val, y_val_pred)
print(f"Validation Accuracy: {val_accuracy * 100:.2f}%")


# Test Accuracy
y_test_pred = logistic_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")